# Figure 2 — Distribution of the target variable

This notebook reproduces **Figure 2** of the manuscript: the distributions of enantiomeric excess (`ee`) and the corresponding free-energy difference (`|ΔΔG‡|`) across the curated dataset of asymmetric organocatalytic Mannich reactions.

## Panel layout

| Panel | Quantity | Representation |
|-------|----------|----------------|
| **A** | `ee`, %  | Histogram (2 % bins) |
| **B** | `\|ΔΔG‡\|`, kcal mol⁻¹ | Histogram (0.2 kcal mol⁻¹ bins) |

## Outputs

The figure is written to the working directory in three formats:

- `figure_02_target_distribution.pdf` — vector, primary submission format
- `figure_02_target_distribution.svg` — vector, editable in Inkscape/Illustrator
- `figure_02_target_distribution.png` — raster, 300 dpi

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, AutoMinorLocator
from scipy import stats

## 2. Matplotlib style and color palette

Settings are tuned for a double-column figure (~178 mm wide) in chemistry journals. `fonttype=42` keeps text editable in the exported PDF and SVG files. The same blue/orange palette is used consistently for `ee` and `|ΔΔG‡|` throughout the manuscript figures.

In [ ]:
plt.rcParams.update({
    'font.family':       'sans-serif',
    'font.sans-serif':   ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size':          9,
    'axes.labelsize':    10,
    'axes.titlesize':    10,
    'axes.linewidth':     0.8,
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'axes.labelpad':      4,
    'xtick.labelsize':    8,
    'ytick.labelsize':    8,
    'xtick.major.width':  0.8,
    'ytick.major.width':  0.8,
    'xtick.minor.width':  0.6,
    'ytick.minor.width':  0.6,
    'xtick.major.size':   3.5,
    'ytick.major.size':   3.5,
    'xtick.minor.size':   2.0,
    'ytick.minor.size':   2.0,
    'xtick.direction':    'out',
    'ytick.direction':    'out',
    'legend.fontsize':    8,
    'legend.frameon':     False,
    'figure.dpi':         120,
    'savefig.dpi':        300,
    'savefig.bbox':      'tight',
    'savefig.pad_inches': 0.05,
    'pdf.fonttype':       42,
    'ps.fonttype':        42,
    'svg.fonttype':      'none',
})

COLOR_EE      = '#4A90C7'  # blue, used for ee
COLOR_DDG     = '#E58A4D'  # orange, used for |ΔΔG‡|
COLOR_NEUTRAL = '#333333'  # dark grey for text and mean line
COLOR_GRID    = '#E0E0E0'  # light grey for gridlines
COLOR_ACCENT  = '#C0392B'  # red for the median line

## 3. Load the dataset

The curated dataset is read from a CSV file. Only two columns are required for this figure: `ee` (enantiomeric excess, %) and `ddG_kcal_mol` (the corresponding |ΔΔG‡| in kcal mol⁻¹).

In [ ]:
DATA_PATH = 'Mannich_dataset.csv'  # adjust path as needed

df = pd.read_csv(DATA_PATH)
ee_values  = df['ee'].to_numpy()
ddg_values = df['ddG_kcal_mol'].to_numpy()

## 4. Descriptive statistics

Summary statistics shown inside the two histogram panels: sample size, mean, median, and skewness for both `ee` and `|ΔΔG‡|`.

In [ ]:
stats_summary = {
    'N':          len(df),
    'ee_mean':    float(np.mean(ee_values)),
    'ee_median':  float(np.median(ee_values)),
    'ee_skew':    float(stats.skew(ee_values)),
    'ddg_mean':   float(np.mean(ddg_values)),
    'ddg_median': float(np.median(ddg_values)),
    'ddg_max':    float(np.max(ddg_values)),
    'ddg_skew':   float(stats.skew(ddg_values)),
}

summary_df = pd.DataFrame({
    'metric': ['N', 'mean', 'median', 'skewness'],
    'ee, %': [
        f"{stats_summary['N']:,}",
        f"{stats_summary['ee_mean']:.1f}",
        f"{stats_summary['ee_median']:.1f}",
        f"{stats_summary['ee_skew']:+.2f}",
    ],
    '|ddG|, kcal/mol': [
        f"{stats_summary['N']:,}",
        f"{stats_summary['ddg_mean']:.2f}",
        f"{stats_summary['ddg_median']:.2f}",
        f"{stats_summary['ddg_skew']:+.2f}",
    ],
})
summary_df

## 5. Helper functions

Two small helpers keep the plotting code uncluttered: a bold panel label and a rounded summary-statistics box.

In [ ]:
def add_panel_label(ax, label, x=-0.18, y=1.08):
    """Add a bold panel label (e.g. 'A') in the upper-left corner."""
    ax.text(x, y, label, transform=ax.transAxes,
            fontsize=12, fontweight='bold', va='top', ha='left')


def add_stat_box(ax, lines, loc='upper right', fontsize=8):
    """Add a rounded white box with multi-line summary statistics."""
    positions = {
        'upper right': (0.97, 0.97, 'right', 'top'),
        'upper left':  (0.03, 0.97, 'left',  'top'),
    }
    x, y, ha, va = positions[loc]
    ax.text(
        x, y, '\n'.join(lines),
        transform=ax.transAxes, ha=ha, va=va,
        fontsize=fontsize, color=COLOR_NEUTRAL, linespacing=1.5,
        bbox=dict(facecolor='white', edgecolor='lightgrey',
                  linewidth=0.5, alpha=0.92,
                  pad=4, boxstyle='round,pad=0.4'),
    )

## 6. Assemble Figure 2

Two side-by-side histograms. Each carries vertical mean/median lines and a corner box collecting *N*, mean, median, and skewness. The y-axes are extended slightly to leave headroom for the annotation boxes.

In [ ]:
fig, (ax_A, ax_B) = plt.subplots(
    1, 2,
    figsize=(7.0, 3.0),                # ~178 x 76 mm, double-column
    gridspec_kw={'wspace': 0.30},
)

ddg_axis_max = np.ceil(stats_summary['ddg_max'] * 2) / 2

# -----------------------------------------------------------------
# Panel A - Histogram of ee
# -----------------------------------------------------------------
ax_A.hist(ee_values, bins=np.arange(0, 101, 2),
          color=COLOR_EE, edgecolor='white', linewidth=0.5, alpha=0.9)
ax_A.axvline(stats_summary['ee_mean'],   color=COLOR_NEUTRAL,
             linestyle='--', linewidth=1.0, alpha=0.75)
ax_A.axvline(stats_summary['ee_median'], color=COLOR_ACCENT,
             linestyle=':',  linewidth=1.4, alpha=0.95)

ax_A.set_xlabel('Enantiomeric excess, % ee')
ax_A.set_ylabel('Count')
ax_A.set_xlim(0, 100)
ax_A.xaxis.set_major_locator(MultipleLocator(20))
ax_A.xaxis.set_minor_locator(MultipleLocator(5))
ax_A.yaxis.set_minor_locator(AutoMinorLocator(2))
ax_A.grid(axis='y', alpha=0.5, linewidth=0.5, color=COLOR_GRID)
ax_A.set_axisbelow(True)
ymin, ymax = ax_A.get_ylim()
ax_A.set_ylim(ymin, ymax * 1.25)

add_stat_box(
    ax_A,
    [
        f'N = {stats_summary["N"]:,}',
        f'Mean = {stats_summary["ee_mean"]:.1f} %',
        f'Median = {stats_summary["ee_median"]:.1f} %',
        f'Skewness = {stats_summary["ee_skew"]:+.2f}',
    ],
    loc='upper left',
)
add_panel_label(ax_A, 'A')

# -----------------------------------------------------------------
# Panel B - Histogram of |ddG|
# -----------------------------------------------------------------
ax_B.hist(ddg_values, bins=np.arange(0, ddg_axis_max + 0.2, 0.2),
          color=COLOR_DDG, edgecolor='white', linewidth=0.5, alpha=0.9)
ax_B.axvline(stats_summary['ddg_mean'],   color=COLOR_NEUTRAL,
             linestyle='--', linewidth=1.0, alpha=0.75)
ax_B.axvline(stats_summary['ddg_median'], color=COLOR_ACCENT,
             linestyle=':',  linewidth=1.4, alpha=0.95)

ax_B.set_xlabel(r'$|\Delta\Delta G^{\ddagger}|$, kcal mol$^{-1}$')
ax_B.set_ylabel('Count')
ax_B.set_xlim(0, ddg_axis_max)
ax_B.xaxis.set_major_locator(MultipleLocator(1.0))
ax_B.xaxis.set_minor_locator(MultipleLocator(0.2))
ax_B.yaxis.set_minor_locator(AutoMinorLocator(2))
ax_B.grid(axis='y', alpha=0.5, linewidth=0.5, color=COLOR_GRID)
ax_B.set_axisbelow(True)
ymin, ymax = ax_B.get_ylim()
ax_B.set_ylim(ymin, ymax * 1.30)

add_stat_box(
    ax_B,
    [
        f'N = {stats_summary["N"]:,}',
        f'Mean = {stats_summary["ddg_mean"]:.2f}',
        f'Median = {stats_summary["ddg_median"]:.2f}',
        f'Skewness = {stats_summary["ddg_skew"]:+.2f}',
    ],
    loc='upper right',
)
add_panel_label(ax_B, 'B')

plt.show()

## 7. Export

In [ ]:
for ext in ('pdf', 'svg', 'png'):
    fig.savefig(f'figure_02_target_distribution.{ext}',
                dpi=300 if ext == 'png' else None)